In [3]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

def plot_observed_county_distribution(
    scenario: str,
    data_dir: str = "../1_clean",
    map_path: str = "../0_map/中国_县_202509_GS(2024)0650.geojson",
    out_dir: str = "output",
    figsize=(7, 7),
    fill_color="#488B87"):

    # ======================
    # 1. Read host data
    # ======================
    data_files = {
        "all": "data_merge_all_20260110.xlsx",
        "china": "data_merge_china.xlsx"
    }

    df = pd.read_excel(Path(data_dir) / data_files[scenario])
    df = df[~(df['host_species'].isna() & df['standard virus name'].isna())].reset_index(drop=True) #key
    print(df.shape[0])
    
    # ======================
    # 2. Clean & summarize
    # ======================
    df = (
        df[df["host_species"].notna()]
        .assign(host_species=lambda x: x["host_species"].str.strip(),
                gb=lambda x: x["gb"].astype(str))
        .groupby("gb", as_index=False)["host_species"]
        .nunique()
        .rename(columns={"host_species": "species_count"})
    )

    # ======================
    # 3. Read map data
    # ======================
    counties = gpd.read_file(map_path)
    counties = counties[counties["name"] != "境界线"].copy()
    counties["gb"] = counties["gb"].astype(str)

    # ======================
    # 4. Merge spatial + data
    # ======================
    gdf = counties.merge(df, on="gb", how="left")
    gdf = gdf[gdf["species_count"].notna()].reset_index(drop=True)

    # Dissolved national boundary (for outline)
    china_outline = counties.dissolve()

    # ======================
    # 5. Plot
    # ======================
    fig, ax = plt.subplots(figsize=figsize)

    china_outline.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1,
        zorder=1
    )

    gdf.plot(
        ax=ax,
        facecolor=fill_color,
        edgecolor="none",
        zorder=2
    )

    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    # ======================
    # 6. Save
    # ======================
    out_path = Path(out_dir) / f"scenario={scenario}"
    out_path.mkdir(parents=True, exist_ok=True)

    date_str = datetime.now().strftime("%Y%m%d")  # 例如 20260107
    fig.savefig(
        out_path / f"{scenario}_county_distribution_{date_str}.tif",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.05
    )
    plt.close(fig)

plot_observed_county_distribution(scenario='all')

47196


In [4]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box
from pathlib import Path
from datetime import datetime

def generate_grid(
    scenario: str,
    cell_size: float,
    land_threshold: float,
    target_crs="EPSG:9822",
    map_path: str = "../0_map/中国_省_202509_GS(2024)0650.geojson",
    out_dir: str = "output"):

    # ======================
    # 1. Read & prepare China boundary
    # ======================
    china = gpd.read_file(map_path)
    china = china[china["name"] != "境界线"].copy()
    china = china.dissolve()  # single polygon
    china_proj = china.to_crs(target_crs)

    # ======================
    # 2. Generate grid (projected CRS)
    # ======================
    minx, miny, maxx, maxy = china_proj.total_bounds

    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)

    grid_cells = [
        box(x, y, x + cell_size, y + cell_size)
        for x in xs
        for y in ys
    ]

    grid_gdf = gpd.GeoDataFrame(
        {
            "grid_index": [f"grid_{i}" for i in range(len(grid_cells))],
            "geometry": grid_cells,
        },
        crs=china_proj.crs,
    )

    # ======================
    # 3. Intersect grid with China
    # ======================
    intersected = gpd.overlay(
        grid_gdf,
        china_proj,
        how="intersection"
    )

    intersected["intersection_area"] = intersected.geometry.area
    intersected["land_fraction"] = intersected["intersection_area"] / (cell_size ** 2)

    # ======================
    # 4. Filter valid grid cells
    # ======================
    valid_ids = (
        intersected.loc[intersected["land_fraction"] >= land_threshold, "grid_index"]
        .unique()
    )

    valid_cells = grid_gdf[grid_gdf["grid_index"].isin(valid_ids)].copy()

    # Convert back to geographic CRS
    valid_cells = valid_cells.to_crs(china.crs)

    # ======================
    # 5. Save outputs
    # ======================
    cell_km = int(cell_size / 1e3)
    out_path = Path(out_dir) / f"scenario={scenario}" / f"cell={cell_km}km"
    out_path.mkdir(parents=True, exist_ok=True)

    valid_cells.to_file(
        out_path / f"{scenario}_{cell_km}km_grid_map.geojson",
        driver="GeoJSON"
    )

    # ======================
    # 6. Plot
    # ======================
    fig, ax = plt.subplots(figsize=(7, 7))

    china.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1,
        zorder=1
    )

    valid_cells.plot(
        ax=ax,
        edgecolor="#BF9895",
        facecolor="none",
        linewidth=0.6,
        zorder=2
    )

    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    date_str = datetime.now().strftime("%Y%m%d")  # 例如 20260107
    fig.savefig(
        out_path / f"{scenario}_{cell_km}km_grid_map_{date_str}.tif",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.05
    )
    plt.close(fig)

    print(
        f"[INFO] {len(valid_cells)} grid cells retained "
        f"(cell={cell_km} km, land ≥ {land_threshold:.0%})"
    )

generate_grid(
    scenario='all',
    cell_size=40000,
    land_threshold=0.5)

[INFO] 6098 grid cells retained (cell=40 km, land ≥ 50%)
